# ESM-2 3B → LangChain DeepAgents, on a Colab A100

End-to-end agentic pipeline for the [`conformers`](https://github.com/dfu99/conformers)
repository: a local ESM-2 3B protein language model, wrapped as LangChain tools, driven by
a DeepAgents orchestrator, reachable from a terminal on your laptop.

## Architecture

```
        your laptop                     Colab A100 VM
   ┌──────────────────┐        ┌──────────────────────────────────┐
   │  cli.py (REPL)   │──HTTP──▶  uvicorn (daemon thread)         │
   │  or curl / ssh   │  tunnel│    ├── FastAPI  /embed /score    │
   └──────────────────┘        │    │            /predict /agent  │
                               │    ├── DeepAgent (Claude)        │
                               │    │     └── @tool wrappers      │
                               │    └── ESM-2 3B  (bf16, ~5.7 GB) │
                               │         └── one threading.Lock   │
                               └──────────────────────────────────┘
```

**The notebook kernel is the only process that may hold the model.** An SSH shell on the
same VM is a *different* process and shares nothing — re-importing the model there would
pull a second 11 GB copy. So every CLI is a thin HTTP client and the kernel owns the GPU.

## Run order

Cells are ordered so that **nothing is imported before it is installed** — that is what
makes the "Restart session" banner unnecessary. Run top to bottom.

| # | Cell | Needs |
|---|------|-------|
| 1 | Install | — |
| 2 | Runtime guard | GPU runtime |
| 3 | Clone repo | network |
| 4 | Load ESM-2 3B | ~11.4 GB disk, ~6 GB VRAM |
| 5 | Inference routines | cell 4 |
| 6 | Repo grounding (αVβ3 genu lock) | cells 3, 5 |
| 7–9 | Tools + DeepAgent + in-notebook loop | `ANTHROPIC_API_KEY` |
| 10–13 | HTTP server, tunnel, SSH, CLI | optional |
| 14 | End-to-end validation | all of the above |

> **Runtime:** `Runtime ▸ Change runtime type ▸ A100 GPU`. The 3B model in bf16 needs
> ~6 GB of VRAM and ~11.4 GB of disk for the fp32 download (the checkpoint ships no
> safetensors, so the full fp32 weights come down and are cast afterwards).


## 1 · Install

First cell on purpose. Colab only requires a restart when pip replaces a module that is
already in `sys.modules`; installing before any heavy import means there is nothing stale.

`torch` is deliberately **not** installed or upgraded — the Colab image ships a build
matched to its CUDA driver, and letting pip resolve a different one is the classic way to
end up on a CPU wheel or a mismatched CUDA build.


In [ ]:
# Installs only. Do not import torch/transformers/numpy in this cell.
# One line on purpose: backslash continuation after a %pip line magic is not dependable.
%pip install -q "deepagents==0.7.9" "transformers>=4.56,<6" accelerate fastapi "uvicorn[standard]" pyngrok requests portpicker fair-esm

# Why these pins:
#   deepagents==0.7.9  -> 0.6.x/0.7.x differ (backend factory removal); the whole 0.7.x
#                         line shares one create_deep_agent signature.
#   transformers>=4.56 -> `dtype=` was added in 4.56. On <=4.55 it is an unknown kwarg
#                         that is silently ignored, and the 3B quietly loads in fp32.
#   accelerate         -> only needed if you pass device_map=; harmless otherwise.
#   fair-esm           -> requested for esm.Alphabet / contact head. NEVER call
#                         esm.pretrained.esm2_t36_3B_UR50D() here: it downloads a SECOND
#                         ~11 GB copy from fbaipublicfiles into $TORCH_HOME and fills the disk.
print("installed")


## 2 · Runtime guard

Fails now, loudly, rather than 40 minutes into a download.

`torch.cuda.is_bf16_supported()` is **not** used: its signature is
`is_bf16_supported(including_emulation=True)`, so it returns `True` on a T4 via emulation
and would wave through exactly the GPU we need to reject. Compute capability ≥ 8 (Ampere)
is the real test.

Nothing about VRAM or disk is hardcoded — Colab serves both A100-40GB and A100-80GB
depending on the High-RAM shape, and Google explicitly declines to publish these numbers.


In [ ]:
import os, shutil, sys

# Must precede the first `import transformers` anywhere in the kernel: huggingface_hub
# freezes its cache paths at import time, so setting this later is a silent no-op.
os.environ.setdefault("HF_HOME", "/content/hf")     # local SSD, never a Drive path

assert sys.version_info >= (3, 11), f"deepagents needs Python >=3.11, got {sys.version}"

import torch, transformers
from importlib.metadata import version

print(f"python       {sys.version.split()[0]}")
print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"deepagents   {version('deepagents')}")
print(f"langchain    {version('langchain')}")

assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > A100 GPU, then re-run from cell 1."
)

cap = torch.cuda.get_device_capability()
name = torch.cuda.get_device_name(0)
free_b, total_b = torch.cuda.mem_get_info()
free_disk = shutil.disk_usage("/content").free / 1e9

print(f"\ngpu          {name}  (sm_{cap[0]}{cap[1]})")
print(f"vram         {free_b/1e9:.1f} GB free / {total_b/1e9:.1f} GB total")
print(f"disk         {free_disk:.1f} GB free on /content")

# Ampere or newer: native bf16, no emulation.
assert cap[0] >= 8, (
    f"{name} (sm_{cap[0]}{cap[1]}) has no native bf16. ESM-2 3B needs an A100/L4/H100."
)
assert free_b / 1e9 > 8, f"Only {free_b/1e9:.1f} GB VRAM free; need ~6 GB for bf16 weights."
if free_disk < 14:
    print(f"\n!! {free_disk:.1f} GB disk free. The fp32 checkpoint download is 11.4 GB.")
print("\nruntime OK")


## 3 · Clone `conformers`

`dfu99/conformers` is public, so this is an anonymous clone — no token, no deploy key, no
`getpass` prompt blocking the notebook.

It is also ~883 MB of git objects. A plain `--depth 1` clone is 1.8 GB and ~3 minutes;
the sparse blobless form below is ~6 MB and ~2 seconds. Only the paths this notebook
actually reads are checked out.

The token path is left in the cell as four commented lines, for the day the repo is
flipped private.


In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/dfu99/conformers.git"
REPO     = pathlib.Path("/content/conformers")
SPARSE   = ["pipelines", "tasks", "results/route_a", "data"]

def sh(*args, **kw):
    return subprocess.run(args, check=True, text=True,
                          capture_output=True, **kw).stdout.strip()

if REPO.exists():
    print("repo present, pulling…")
    print(sh("git", "-C", str(REPO), "pull", "--ff-only") or "up to date")
else:
    # blobless + sparse: fetch trees, not the 883 MB of blobs, then check out 4 paths.
    sh("git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", REPO_URL, str(REPO))
    sh("git", "-C", str(REPO), "sparse-checkout", "set", *SPARSE)
    print(f"cloned {REPO}")

# If the repo is ever made private, swap the clone line for:
#   from google.colab import userdata            # Colab Secrets, not getpass
#   tok = userdata.get("GH_TOKEN")
#   sh("git","clone",...,f"https://{tok}@github.com/dfu99/conformers.git",str(REPO))
#   sh("git","-C",str(REPO),"remote","set-url","origin",REPO_URL)  # scrub .git/config

SCRIPTS = REPO / "pipelines" / "route_a" / "scripts"
for p in (str(REPO), str(SCRIPTS)):
    if p not in sys.path:
        sys.path.append(p)

# NOTE on importing repo code: the route_a scripts import each other FLAT
# (lock_energy_analysis.py does `import snapback_md as sb`), so /content/conformers alone
# is not enough -- the scripts dir has to be on sys.path too, which is why it is appended
# above. Those modules also need openmm/pdbfixer, which this notebook does not install.
# This notebook READS the repo's data files; it does not import its MD modules.
# To actually run one, shell out with cwd at the repo root so `results/...` resolves:
#   subprocess.run([sys.executable, "pipelines/route_a/scripts/snapback_md.py", "--smoke"],
#                  cwd=REPO, env={**os.environ, "PYTHONPATH": str(SCRIPTS)})

print(f"{sum(1 for _ in REPO.rglob('*.py'))} python files checked out")
print("sys.path +=", REPO, "and", SCRIPTS)


## 4 · Load ESM-2 3B in bfloat16

One model object serves both jobs. `EsmForMaskedLM` contains the encoder as `.esm`, so
aliasing it gives per-residue embeddings and masked-LM logits from a single 5.7 GB copy
of the weights.

Two traps avoided here:

- `EsmModel.from_pretrained()` defaults to `add_pooling_layer=True`, which **randomly
  initializes** a pooler that exists in no ESM-2 checkpoint. `pooler_output` is then
  meaningless noise. Going through `EsmForMaskedLM.esm` sidesteps it entirely.
- `device_map=` raises `ValueError` without `accelerate`, and buys nothing for a model
  that fits one GPU. Plain `.to("cuda")`.

`HF_HOME` was set back in the runtime-guard cell, before the first `transformers` import —
`huggingface_hub` freezes its cache paths at import time, so setting it here would be a
silent no-op. It points at local SSD; a mounted Drive path turns the 11.4 GB pull into
30+ minutes and can time out.


In [ ]:
import time, torch
from transformers import AutoTokenizer, EsmForMaskedLM

MODEL_ID = "facebook/esm2_t36_3B_UR50D"
DEVICE   = "cuda"

t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
mlm = EsmForMaskedLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to(DEVICE).eval()
enc = mlm.esm                    # pooler-free encoder, same weights, no extra VRAM

EMBED_DIM = mlm.config.hidden_size
N_LAYERS  = mlm.config.num_hidden_layers

print(f"loaded in {time.time()-t0:.0f}s")
print(f"layers {N_LAYERS} | dim {EMBED_DIM} | params {sum(p.numel() for p in mlm.parameters())/1e9:.2f} B")
print(f"vram   {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

# Tokenizer geometry -- the single most important fact for mutation scoring.
# <cls>(0) is prepended and <eos>(2) appended, so TOKEN INDEX == 1-BASED RESIDUE POSITION.
_probe = tok("MKTV")["input_ids"]
assert _probe[0] == tok.cls_token_id and _probe[-1] == tok.eos_token_id
assert len(_probe) == 6, _probe
print(f"\ntok('MKTV') = {_probe} -> {tok.convert_ids_to_tokens(_probe)}")
print("token index == 1-based residue position")

# 13 of the 33 vocab entries are not amino acids; an unfiltered argmax returns '<null_1>'.
CANON  = "ACDEFGHIKLMNPQRSTVWY"
AA_ID  = {a: tok.convert_tokens_to_ids(a) for a in CANON}
ID_AA  = {i: a for a, i in AA_ID.items()}
assert len(set(AA_ID.values())) == 20 and tok.unk_token_id not in AA_ID.values()
print(f"canonical AA token ids: {min(AA_ID.values())}..{max(AA_ID.values())}")

# ESM-2 uses ROTARY position embeddings -- max_position_embeddings=1026 is not a length
# limit and there is no silent truncation. Sequences of 1000-2500 forward in one pass;
# cost is O(L^2) attention, not a hard cap.
print(f"position_embedding_type = {mlm.config.position_embedding_type}")


## 5 · Inference routines

Three primitives. Every one of them runs under a single `threading.Lock`, because the
FastAPI server in cell 10 hands `def` routes to anyio's threadpool and will otherwise run
several forward passes into the same GPU concurrently.

**Scoring conventions.** Both are implemented because they disagree sharply — on one test
position the same R→K substitution scores −3.71 by wt-marginals and −0.09 by
masked-marginals:

- `wt` — one forward pass on the unmasked sequence; `log p(mut) − log p(wt)` at the site.
  Cost: 1 pass, any number of mutations.
- `masked` — mask the site first, then the same difference. This is the ESM-1v convention
  (Meier et al. 2021) and the more accurate one. Cost: one pass per distinct *position*
  (not per mutation), batched.

Sign convention: **higher = more tolerated**, negative = predicted deleterious.

`.float()` before `log_softmax` matters — bf16 log-softmax over the 33-token vocab loses
2–3 decimals exactly on the low-probability tail where a deleterious score lives.


In [ ]:
import re, threading, torch

GPU_LOCK = threading.Lock()          # one forward pass at a time; see cell 10
MUT_RE   = re.compile(r"^([A-Z])(\d+)([A-Z])$")


@torch.inference_mode()
def extract_embeddings(sequences, batch_size=4, per_residue=False):
    """Mean-pooled sequence embeddings (and optionally per-residue ones).

    Pooling is attention-mask aware and excludes <cls>/<eos>/<pad>. ESM-2 sets
    token_dropout=True, which rescales embeddings by a per-row factor computed from
    attention_mask.sum(-1) -- so a padded batch WITHOUT the mask is silently wrong.
    """
    if isinstance(sequences, str):
        sequences = [sequences]
    seq_vecs, res_vecs = [], []
    with GPU_LOCK:
        for i in range(0, len(sequences), batch_size):
            chunk = sequences[i:i + batch_size]
            b = tok(chunk, return_tensors="pt", padding=True).to(DEVICE)
            # .float() matters: in bf16 the pooling divisor for a 927-residue chain
            # rounds 927 -> 928, a length-dependent scale error on every embedding.
            h = enc(**b).last_hidden_state.float()               # (B, T, 2560)

            m = b["attention_mask"].clone()
            m[:, 0] = 0                                          # drop <cls>
            eos = b["attention_mask"].sum(1) - 1
            m[torch.arange(m.size(0), device=DEVICE), eos] = 0   # drop <eos>

            mf = m.unsqueeze(-1).to(h.dtype)
            pooled = (h * mf).sum(1) / mf.sum(1).clamp(min=1)
            for j, s in enumerate(chunk):
                seq_vecs.append(pooled[j].cpu())
                if per_residue:
                    res_vecs.append(h[j, 1:1 + len(s)].cpu())
    return (seq_vecs, res_vecs) if per_residue else seq_vecs


@torch.inference_mode()
def score_mutations(sequence, mutations, method="masked", batch_size=8):
    """Zero-shot variant scores. mutations like ["K459A", "E598A"], 1-based.

    Returns higher = more tolerated. Raises if the WT letter does not match the sequence,
    which is the guard that catches a numbering mismatch instead of returning plausible
    garbage.
    """
    parsed = []
    for m in mutations:
        hit = MUT_RE.match(m.strip().upper())
        if not hit:
            raise ValueError(f"bad mutation {m!r}; expected e.g. 'K459A'")
        wt, pos, mut = hit.group(1), int(hit.group(2)), hit.group(3)
        if not 1 <= pos <= len(sequence):
            raise ValueError(f"{m}: position {pos} outside 1..{len(sequence)}")
        if sequence[pos - 1] != wt:
            raise ValueError(
                f"{m}: sequence has {sequence[pos-1]} at {pos}, not {wt}. "
                "Wrong numbering or wrong chain."
            )
        if wt not in AA_ID or mut not in AA_ID:
            raise ValueError(f"{m}: non-canonical amino acid")
        parsed.append((m, wt, pos, mut))

    positions = sorted({p for _, _, p, _ in parsed})
    logp = {}
    with GPU_LOCK:
        if method == "wt":
            b = tok(sequence, return_tensors="pt").to(DEVICE)
            row = torch.log_softmax(mlm(**b).logits[0].float(), dim=-1)
            for p in positions:
                logp[p] = row[p]                      # token index == 1-based position
        elif method == "masked":
            for i in range(0, len(positions), batch_size):
                chunk = positions[i:i + batch_size]
                b = tok([sequence] * len(chunk), return_tensors="pt",
                        padding=True).to(DEVICE)
                for r, p in enumerate(chunk):
                    b["input_ids"][r, p] = tok.mask_token_id
                lg = mlm(**b).logits.float()
                for r, p in enumerate(chunk):
                    logp[p] = torch.log_softmax(lg[r, p], dim=-1)
        else:
            raise ValueError("method must be 'masked' or 'wt'")

    out = []
    for m, wt, pos, mut in parsed:
        row = logp[pos]
        lp_wt, lp_mut = float(row[AA_ID[wt]]), float(row[AA_ID[mut]])
        out.append({"mutation": m, "position": pos, "wt": wt, "mut": mut,
                    "method": method, "score": round(lp_mut - lp_wt, 4),
                    "logp_wt": round(lp_wt, 4), "logp_mut": round(lp_mut, 4)})
    return out


@torch.inference_mode()
def predict_masked_residues(sequence, positions, top_k=5):
    """Top-k canonical amino acids at each masked position (1-based)."""
    positions = sorted({int(p) for p in positions})
    for p in positions:
        if not 1 <= p <= len(sequence):
            raise ValueError(f"position {p} outside 1..{len(sequence)}")
    aa_ids = torch.tensor([AA_ID[a] for a in CANON], device=DEVICE)

    out = []
    with GPU_LOCK:
        for i in range(0, len(positions), 8):
            chunk = positions[i:i + 8]
            b = tok([sequence] * len(chunk), return_tensors="pt", padding=True).to(DEVICE)
            for r, p in enumerate(chunk):
                b["input_ids"][r, p] = tok.mask_token_id
            lg = mlm(**b).logits.float()
            for r, p in enumerate(chunk):
                row = torch.log_softmax(lg[r, p], dim=-1)
                probs = row[aa_ids].exp()                   # 20 canonical only
                top = torch.topk(probs, min(top_k, 20))
                out.append({
                    "position": p,
                    "wt": sequence[p - 1],
                    "top_k": [{"aa": CANON[j], "prob": round(float(v), 4)}
                              for v, j in zip(top.values, top.indices)],
                })
    return out


# --- self-check: the logic above is worth exactly one assert -----------------
_S = "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLA"
_e = extract_embeddings(_S)
assert _e[0].shape == (EMBED_DIM,), _e[0].shape

# Does the attention_mask actually keep <pad> out of the mean? The padded row is the SHORT
# one -- the batch pads to the longest, so _S itself is never padded and comparing it across
# batch shapes would only measure bf16 kernel noise (~0.06 elementwise on the 3B, which is
# why an allclose(atol=1e-5) here fails on real hardware). Compare the short sequence pooled
# inside the batch against the same sequence pooled alone, by DIRECTION: averaging in 33 pad
# embeddings would swing the vector hard, while bf16 noise leaves the direction intact.
_short = _S[:30]
_b = extract_embeddings([_S, _short])
_alone = extract_embeddings(_short)[0]
_cos = float(torch.nn.functional.cosine_similarity(_b[1], _alone, dim=0))
_nrm = float(_b[1].norm() / _alone.norm())
# Both halves are needed. Cosine catches pads with a non-zero hidden state dragging the
# direction; it is blind to scale, so a wrong DIVISOR (dividing by 65 instead of 32, or
# bf16 rounding 927 -> 928) would slip past it. The norm ratio catches exactly that, and
# 5% leaves room for bf16 kernel noise, which moves the norm well under 1%.
assert _cos > 0.999, f"padded pooling leaked pad tokens: cos={_cos:.6f}"
assert abs(_nrm - 1.0) < 0.05, f"pooling divisor is wrong: norm ratio={_nrm:.4f}"
print(f"padding-mask check: cos={_cos:.6f}  norm_ratio={_nrm:.4f}")

# The WT-letter guard must fire on a wrong letter -- this is the numbering tripwire.
try:
    score_mutations(_S, ["A2G"]); raise SystemExit("WT-letter guard did not fire")
except ValueError as err:
    assert "not A" in str(err), err

# Assert what this code owns (indexing, bookkeeping), NOT what the model believes. An
# assert on the SIGN of a score gates the notebook on a number that varies by checkpoint:
# K2A sits right after the initiator Met, where the masked marginal is flattest, and it
# scores POSITIVE on the 35M/150M/650M checkpoints.
_sc = score_mutations(_S, ["K2A"], method="masked")[0]
assert (_sc["position"], _sc["wt"], _sc["mut"]) == (2, "K", "A"), _sc
assert abs(_sc["score"] - (_sc["logp_mut"] - _sc["logp_wt"])) < 1e-3, _sc
assert _sc["logp_wt"] < 0 and _sc["logp_mut"] < 0, _sc     # log-probs, never positive
print("self-check passed:", _sc)


## 6 · Repo grounding — the αVβ3 genu lock

This is what makes the pipeline a test of *this* project rather than a demo.

`results/route_a/extended_state_b.pdb` is the extended-state αVβ3 ectodomain the snap-back
MD runs on. Chain A (αV) is 927 residues, **contiguous 1–927 in the file's own numbering**,
so the sequence read straight out of the PDB indexes identically to the residue numbers in
`snapback_md.py`. No UniProt mapping, no offset guesswork.

That matters because the numbering is model-specific: `snapback_md.py` documents that these
are *not* 1JV2 numbers, and getting it wrong is called out in the repo as "this repo's most
repeated bug".

The four cross-knee salt bridges `snapback_md.py` monitors:

| bridge | anion | cation |
|---|---|---|
| E598–K459 | A:598 | A:459 |
| D457–K688 | A:457 | A:688 |
| K459–E636 | A:636 | A:459 |
| D595–K688 | A:595 | A:688 |

> **This is not wild-type αV.** The file renumbers 1JV2's *observed* residues contiguously,
> so αV 839–867 (disordered, absent) is spliced out and everything after resSeq 838 is
> offset by +29. ESM-2 is therefore scoring a 29-residue deletion variant with an
> artificial junction at 838|839. All six genu sites are ≤ 688 — inside the identity
> segment, >150 residues from the junction — so their numbering is exact and their local
> sequence context is intact, but the scores are not directly comparable to ones computed
> on full-length UniProt P06756. Chain B is offset too (β3 mature 55–434, then +151).


In [ ]:
import json, pathlib

PDB  = REPO / "results" / "route_a" / "extended_state_b.pdb"
LOCK = REPO / "results" / "route_a" / "lock_energy.json"

THREE2ONE = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
    "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
    "THR":"T","TRP":"W","TYR":"Y","VAL":"V","HID":"H","HIE":"H","HIP":"H","CYX":"C",
}

def chain_sequences(pdb_path):
    """One-letter sequence per chain, in file order, with the residue numbers."""
    chains, seen = {}, set()
    for line in pdb_path.read_text().splitlines():
        if not line.startswith("ATOM"):
            continue
        ch, num, res = line[21], int(line[22:26]), line[17:20].strip()
        if (ch, num) in seen:
            continue
        seen.add((ch, num))
        chains.setdefault(ch, []).append((num, THREE2ONE.get(res, "X")))
    return {ch: ("".join(a for _, a in rs), [n for n, _ in rs]) for ch, rs in chains.items()}

CHAINS = chain_sequences(PDB)
AV_SEQ, AV_NUMS = CHAINS["A"]          # alphaV
B3_SEQ, B3_NUMS = CHAINS["B"]          # beta3

for ch, (seq, nums) in CHAINS.items():
    contiguous = nums == list(range(nums[0], nums[-1] + 1))
    print(f"chain {ch}: {len(seq)} residues, {nums[0]}-{nums[-1]}, contiguous={contiguous}")

# The whole grounding rests on this: residue number == 1-based sequence index.
assert AV_NUMS == list(range(1, len(AV_SEQ) + 1)), "chain A is not contiguous from 1"

# PROVENANCE -- read before interpreting any score. This file renumbers 1JV2's OBSERVED
# residues contiguously, so it is NOT the wild-type sequence (snapback_md.py:102-106):
#   chain A  1-838    -> aV mature 1-838    (identity; aV 839-867 is disordered/absent)
#            839-927  -> aV mature 868-956  (+29)
#   chain B  1-380    -> b3 mature 55-434   (+54; b3 435-531 absent)
#            381-539  -> b3 mature 532-690  (+151)
# So AV_SEQ carries an artificial 838|839 junction and ESM-2 is scoring a 29-residue
# deletion variant. All six genu sites are <= 688, i.e. inside the identity segment and
# >150 residues from the junction, so their numbering is exact and their local context is
# intact -- but these scores are not directly comparable to ones computed on full-length
# UniProt P06756.
AV_JUNCTION = 838

# Genu-lock salt-bridge partners, from snapback_md.py SALT_PAIRS.
GENU_SITES = {457: "D", 459: "K", 595: "D", 598: "E", 636: "E", 688: "K"}
assert max(GENU_SITES) < AV_JUNCTION, "a genu site crossed the numbering junction"

for pos, expect in GENU_SITES.items():
    got = AV_SEQ[pos - 1]
    assert got == expect, f"A:{pos} is {got}, expected {expect} -- numbering drift"
print(f"\nall {len(GENU_SITES)} genu-lock residues match snapback_md.py numbering")

# MD lock energies (obj-081), for the comparison in cell 14.
LOCK_ENERGY = json.loads(LOCK.read_text())

# lock_energy.json ships per_mutation for only K459A / E598A / the double. Derive the
# same quantity for all six sites from per_bridge: a site's lock energy is the sum over
# the bridges it takes part in, as either the anion or the cation.
MD_LOCK_KCAL, MD_BRIDGES = {}, {}
for bridge, d in LOCK_ENERGY["per_bridge"].items():
    for role in ("anion", "cation"):
        pos = int(d[role][1:])                      # "A598" -> 598
        MD_LOCK_KCAL[pos] = MD_LOCK_KCAL.get(pos, 0.0) + abs(d["E_total_ddd_kcal"])
        MD_BRIDGES.setdefault(pos, []).append(bridge)

# The derivation must reproduce the two single mutants the repo already computed.
for mut, pos in (("K459A", 459), ("E598A", 598)):
    want = LOCK_ENERGY["per_mutation"][mut]["lock_energy_removed_ddd_kcal"]
    got = round(MD_LOCK_KCAL[pos], 2)
    assert abs(got - want) < 0.01, f"{mut}: derived {got}, lock_energy.json says {want}"
print(f"\nper-site derivation reproduces lock_energy.json for K459A and E598A")

print("\nMD lock energy removed by Ala substitution (kcal/mol, dist-dependent dielectric):")
for pos in sorted(MD_LOCK_KCAL, key=lambda p: -MD_LOCK_KCAL[p]):
    print(f"  {AV_SEQ[pos-1]}{pos}A  {MD_LOCK_KCAL[pos]:6.2f}   "
          f"breaks {', '.join(MD_BRIDGES[pos])}")

GENU_MUTATIONS = [f"{aa}{pos}A" for pos, aa in sorted(GENU_SITES.items())]
print("\nmutations to score:", GENU_MUTATIONS)
print(f"alphaV chain is {len(AV_SEQ)} residues -> {len(AV_SEQ)+2} tokens, one forward pass")


## 7 · LangChain tool wrappers

Four `@tool`s over the three primitives — the fourth consumes the handles the first one
hands out, so the store is readable rather than write-only. Two rules shape them:

1. **Never return the raw array.** A 2560-float list serializes to ~20,200 characters —
   about 15k tokens of context *per call*. The embedding stays in a module-level dict and
   the tool returns a handle plus summary statistics.
2. **Never return numpy/torch objects.** A dict containing `np.float32` silently falls back
   to `str(dict)`, so the model receives `{'v': array([0., 0.])}` instead of JSON. Everything
   is cast to plain `float`/`int`/`str` on the way out.

`parse_docstring=True` is what turns the `Args:` block into per-argument schema descriptions;
without it the whole docstring is dumped raw into the tool description and the arg
descriptions never reach the model.


In [ ]:
from langchain_core.tools import tool     # NOT langchain_core.pydantic_v1 (gone in 1.x)

EMBEDDING_STORE = {}                      # handle -> tensor, stays out of the context window


@tool(parse_docstring=True)
def esm_extract_embeddings(sequence: str, name: str = "seq",
                           per_residue: bool = False) -> dict:
    """Embed a protein sequence with ESM-2 3B and store the vector locally.

    Returns a handle and summary statistics, never the raw vector. Pass the handle to
    esm_compare_embeddings to compare two sequences.

    Args:
        sequence: Single-letter amino-acid sequence, e.g. "MKTVRQ...".
        name: Short label used as the storage handle.
        per_residue: Also store the per-residue matrix (n_residues x 2560) under
            "<handle>:res". Only the shape is returned.
    """
    sequence = sequence.strip().upper()
    if not sequence or set(sequence) - set(CANON):
        return {"error": f"non-canonical residues: {sorted(set(sequence) - set(CANON))}"}
    handle = f"{name}:{len(EMBEDDING_STORE)}"
    if per_residue:
        (vec,), (res,) = extract_embeddings(sequence, per_residue=True)
        EMBEDDING_STORE[handle + ":res"] = res
    else:
        vec, res = extract_embeddings(sequence)[0], None
    EMBEDDING_STORE[handle] = vec
    out = {
        "handle": handle,
        "n_residues": len(sequence),
        "embedding_dim": int(vec.shape[0]),
        "l2_norm": round(float(vec.norm()), 4),
        "mean": round(float(vec.mean()), 5),
        "first_8_dims": [round(float(x), 4) for x in vec[:8]],
    }
    if res is not None:
        out["per_residue_handle"] = handle + ":res"
        out["per_residue_shape"] = list(res.shape)
    return out


@tool(parse_docstring=True)
def esm_compare_embeddings(handle_a: str, handle_b: str) -> dict:
    """Cosine similarity between two embeddings already stored by esm_extract_embeddings.

    Args:
        handle_a: Handle returned by esm_extract_embeddings.
        handle_b: Handle returned by esm_extract_embeddings.
    """
    missing = [h for h in (handle_a, handle_b) if h not in EMBEDDING_STORE]
    if missing:
        return {"error": f"unknown handle(s) {missing}; stored: {list(EMBEDDING_STORE)}"}
    a, b = EMBEDDING_STORE[handle_a], EMBEDDING_STORE[handle_b]
    if a.shape != b.shape:
        return {"error": f"shape mismatch {list(a.shape)} vs {list(b.shape)}"}
    cos = torch.nn.functional.cosine_similarity(a, b, dim=0)
    return {"handle_a": handle_a, "handle_b": handle_b,
            "cosine_similarity": round(float(cos), 6)}


@tool(parse_docstring=True)
def esm_score_mutations(sequence: str, mutations: list[str],
                        method: str = "masked") -> dict:
    """Zero-shot fitness scores for point mutations, using ESM-2 3B log-likelihoods.

    Score is log p(mutant) - log p(wild-type) at the site. Higher means better tolerated;
    negative means predicted deleterious. Fails loudly if a wild-type letter does not
    match the sequence, which means the numbering is wrong.

    Args:
        sequence: Single-letter amino-acid sequence.
        mutations: Mutations as WT-position-MUT, 1-based, e.g. ["K459A", "E598A"].
        method: "masked" for ESM-1v masked-marginals (accurate, one pass per position)
            or "wt" for wild-type marginals (one pass total, cruder).
    """
    try:
        scored = score_mutations(sequence.strip().upper(), mutations, method=method)
    except ValueError as e:
        return {"error": str(e)}
    ranked = sorted(scored, key=lambda d: d["score"])
    return {
        "method": method,
        "n_scored": len(scored),
        "scores": scored,
        "most_deleterious": ranked[0]["mutation"],
        "least_deleterious": ranked[-1]["mutation"],
    }


@tool(parse_docstring=True)
def esm_predict_masked_residues(sequence: str, positions: list[int],
                                top_k: int = 5) -> dict:
    """Predict which amino acids ESM-2 3B expects at given positions.

    Each position is masked in turn and the top-k canonical amino acids are returned with
    their probabilities. Useful for asking what the model thinks a site is "for".

    Args:
        sequence: Single-letter amino-acid sequence.
        positions: 1-based positions to mask and predict.
        top_k: How many amino acids to return per position (max 20).
    """
    try:
        preds = predict_masked_residues(sequence.strip().upper(), positions,
                                        top_k=max(1, min(int(top_k), 20)))
    except ValueError as e:
        return {"error": str(e)}
    return {"n_positions": len(preds), "predictions": preds}


TOOLS = [esm_extract_embeddings, esm_score_mutations, esm_predict_masked_residues,
         esm_compare_embeddings]
for t in TOOLS:
    print(f"{t.name:<30} args={list(t.args_schema.model_json_schema()['properties'])}")

_probe = esm_extract_embeddings.invoke(
    {"sequence": _S, "name": "probe", "per_residue": True})
assert _probe["embedding_dim"] == EMBED_DIM and "first_8_dims" in _probe
assert _probe["per_residue_shape"] == [len(_S), EMBED_DIM], _probe
_same = esm_compare_embeddings.invoke(
    {"handle_a": _probe["handle"], "handle_b": _probe["handle"]})
assert abs(_same["cosine_similarity"] - 1.0) < 1e-4, _same
print("\ntool dispatch OK:", _probe["handle"], "| per-residue",
      _probe["per_residue_shape"], "| self-cosine", _same["cosine_similarity"])


## 8 · DeepAgent

Put your key in **Colab Secrets** (🔑 in the left sidebar) as `ANTHROPIC_API_KEY` and
enable notebook access. Secrets are the only non-blocking way to get a credential into an
unattended notebook — `getpass` stalls forever with no tty, and a literal key in a cell
gets committed.

Three API facts, all of which differ from most tutorials on the web:

- The parameter is **`system_prompt=`**, not `instructions=`. `instructions=` was the
  0.0.x spelling and now raises `TypeError`.
- **`async_create_deep_agent` does not exist** — it was removed after 0.0.x. Use
  `create_deep_agent` + `await agent.ainvoke()`.
- Passing `model=None` emits a deprecation warning and silently defaults to a Sonnet
  build; passing `system_prompt=None` sends a literally **empty** system prompt, because
  0.7 deleted its built-in base prompt. Both are always passed explicitly here.

The model is built through `init_chat_model` rather than a `"provider:model"` string so
that `max_tokens` can be pinned — the default resolves from the model profile to 64000,
which is a lot of output to pay for on every turn of an agent loop.

Note the agent also gets **8 built-in tools** (`ls`, `read_file`, `write_file`, `edit_file`,
`delete`, `glob`, `grep`, `task`) on top of ours; `tools=` is additive and cannot remove
them. That is why the system prompt explicitly steers it to the `esm_*` tools.


In [ ]:
import os

try:
    from google.colab import userdata          # not on PyPI; guard for non-Colab runs
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass                                        # running outside Colab; use the ambient env
except Exception as e:
    raise RuntimeError(
        "Add ANTHROPIC_API_KEY in Colab Secrets (key icon, left sidebar) and toggle "
        f"notebook access on. Underlying error: {e}"
    )

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY is not set"

from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model

ORCH_MODEL = "anthropic:claude-opus-5"     # "anthropic:claude-sonnet-5" is the cheaper tier

SYSTEM_PROMPT = """You are a protein-engineering assistant for the `conformers` project,
which studies integrin (alphaVbeta3, alpha5beta1) conformational change.

You have a local ESM-2 3B protein language model on an A100, exposed as four tools:
  esm_extract_embeddings      - embed a sequence, returns a handle + statistics
  esm_score_mutations         - zero-shot variant scores, log p(mut) - log p(wt)
  esm_predict_masked_residues - what the model expects at a masked position
  esm_compare_embeddings      - cosine similarity between two stored handles

Always prefer these tools over the generic filesystem tools for anything about sequences,
mutations, or embeddings. Never invent a score: if a tool returns an error, report it.

Interpreting scores: they are evolutionary-fitness proxies learned from sequence data.
They measure how surprising a substitution is, NOT its mechanical or energetic effect on a
structure. Say so when a user compares them to a physics-based quantity.

Context: the alphaV genu lock is a cross-knee salt-bridge network (E598-K459, D457-K688,
K459-E636, D595-K688) that holds the extended state open. Residue numbers refer to
results/route_a/extended_state_b.pdb, which is 1-based and contiguous -- but that file is a
1JV2-derived model with alphaV 839-867 spliced out, so it is a deletion variant, not
wild-type alphaV. Say so if a user treats a score as applying to wild-type alphaV.

Be concise. Show the numbers you got back."""

# NO temperature/top_p/top_k: sampling params are REMOVED on Opus 5 / Sonnet 5 and return
# HTTP 400. Nothing validates this client-side, so it would 400 on the first turn, not here.
# The depth knob on these models is effort, not temperature.
llm = init_chat_model(ORCH_MODEL, max_tokens=16000)

print(f"model ready: {type(llm).__name__} on {ORCH_MODEL}")
print("note: a bad model id is NOT validated client-side; it 404s on the first turn.")
# The agent graph itself is built in the next cell, with a checkpointer, as CHAT.


## 9 · In-notebook agent loop

`stream_mode="messages"` is the mode that yields LLM tokens as they arrive. Two details:

- It also emits `ToolMessage` objects carrying full tool results, so filtering on
  `msg.type == "AIMessageChunk"` (the literal string — `AIMessage.type` is `"ai"`) is what
  keeps a tool payload from being spliced into the middle of the prose.
- Print `msg.text`, never `msg.content`. With Anthropic, `content` is a list of blocks
  (text / thinking / tool_use) and `sys.stdout.write(msg.content)` raises `TypeError` on
  the first thinking block. `.text` is a property that flattens both forms.

An `InMemorySaver` gives the loop conversational memory. It dies with the kernel — if the
runtime restarts, every thread silently resets to an empty conversation with no error.


In [ ]:
import sys, uuid
from langgraph.checkpoint.memory import InMemorySaver

CHAT = create_deep_agent(
    model=llm, tools=TOOLS, system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)
THREAD = {"configurable": {"thread_id": str(uuid.uuid4())}}


def ask(question, thread=None, stream=True):
    """Send one turn to the agent. Streams assistant tokens; returns the final text."""
    cfg = thread or THREAD
    payload = {"messages": [{"role": "user", "content": question}]}
    if not stream:
        result = CHAT.invoke(payload, cfg)
        ask.last_tools = [m.name for m in result["messages"] if m.type == "tool"]
        return result["messages"][-1].text

    parts, fired = [], []
    for msg, _meta in CHAT.stream(payload, cfg, stream_mode="messages"):
        if msg.type == "AIMessageChunk":            # not "ai" -- that is AIMessage
            chunk = msg.text
            if chunk:
                parts.append(chunk)
                sys.stdout.write(chunk)
                sys.stdout.flush()
        elif msg.type == "tool":
            fired.append(msg.name)
            sys.stdout.write(f"\n  [tool {msg.name} -> {len(str(msg.content))} chars]\n")
    print()
    ask.last_tools = fired          # cell 14 asserts the agent really dispatched a tool
    return "".join(parts)


_ = ask(
    f"Score K459A and E598A on this alphaV sequence using masked-marginals, then say "
    f"which the model finds more surprising:\n{AV_SEQ}"
)


## 10 · HTTP server

The kernel owns the model, so the kernel serves it. `uvicorn` runs in a daemon thread —
`Server.run()` calls `capture_signals()`, which is a no-op off the main thread, and that is
exactly why the pattern works. The side effect is that **interrupt-kernel will not stop the
server**; only `_server.should_exit = True` does, which is what the re-run guard uses.

`nest_asyncio` is deliberately not installed: the daemon thread builds its own event loop,
so it is unnecessary, and it monkeypatches `asyncio` globally in a way that can break other
async libraries in the same kernel.

**The lock is not optional.** FastAPI runs `def` routes on anyio's threadpool, so a single
uvicorn worker still admits several handlers at once. Every route below goes through
`GPU_LOCK`. And never `workers>1` — each worker is a separate process that would load its
own copy of the weights and OOM the card.


In [ ]:
import json, secrets, threading, time
import portpicker, uvicorn
from fastapi import Depends, FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel, Field

# Re-running this cell without shutting the old server down gives EADDRINUSE *inside the
# daemon thread*, where it is swallowed -- the port keeps serving the OLD app object.
if "_server" in globals():
    _server.should_exit = True
    _thread.join(timeout=5)
    print("previous server stopped")

API_TOKEN = secrets.token_urlsafe(24)
PORT      = portpicker.pick_unused_port()
MAX_LEN   = 2500                      # O(L^2) attention; a guard, not a model limit

app = FastAPI(title="conformers ESM-2 3B agent")
bearer = HTTPBearer()


def auth(cred: HTTPAuthorizationCredentials = Depends(bearer)):
    # .encode(): compare_digest on str rejects non-ASCII, turning a junk header into a 500
    if not secrets.compare_digest(cred.credentials.encode("utf-8", "ignore"),
                                  API_TOKEN.encode()):
        raise HTTPException(401, "bad token")


class SeqReq(BaseModel):
    sequence: str
    name: str = "seq"

class ScoreReq(BaseModel):
    sequence: str
    mutations: list[str]
    method: str = Field("masked", pattern="^(masked|wt)$")

class MaskReq(BaseModel):
    sequence: str
    positions: list[int]
    top_k: int = 5

class AgentReq(BaseModel):
    message: str
    thread_id: str = "cli"


def _check(seq: str) -> str:
    seq = seq.strip().upper()
    if len(seq) > MAX_LEN:
        raise HTTPException(413, f"sequence longer than {MAX_LEN}")
    bad = set(seq) - set(CANON)
    if bad:
        raise HTTPException(422, f"non-canonical residues: {sorted(bad)}")
    return seq


@app.get("/health")
def health():                                    # unauthenticated, for tunnel readiness
    return {"ok": True, "model": MODEL_ID, "gpu": torch.cuda.get_device_name(0),
            "vram_free_gb": round(torch.cuda.mem_get_info()[0] / 1e9, 2)}


@app.post("/embed", dependencies=[Depends(auth)])
def embed(r: SeqReq):
    return esm_extract_embeddings.invoke({"sequence": _check(r.sequence), "name": r.name})


@app.post("/score", dependencies=[Depends(auth)])
def score(r: ScoreReq):
    return esm_score_mutations.invoke(
        {"sequence": _check(r.sequence), "mutations": r.mutations, "method": r.method})


@app.post("/predict", dependencies=[Depends(auth)])
def predict(r: MaskReq):
    return esm_predict_masked_residues.invoke(
        {"sequence": _check(r.sequence), "positions": r.positions, "top_k": r.top_k})


@app.post("/agent", dependencies=[Depends(auth)])
def agent(r: AgentReq):
    """NDJSON stream: one JSON object per line. Simpler than SSE and one less dependency."""
    cfg = {"configurable": {"thread_id": r.thread_id}}

    def gen():
        payload = {"messages": [{"role": "user", "content": r.message}]}
        try:
            for msg, _meta in CHAT.stream(payload, cfg, stream_mode="messages"):
                if msg.type == "AIMessageChunk" and msg.text:
                    yield json.dumps({"t": "token", "v": msg.text}) + "\n"
                elif msg.type == "tool":
                    yield json.dumps({"t": "tool", "name": msg.name}) + "\n"
        except Exception as e:                    # never leave the client hanging
            yield json.dumps({"t": "error", "v": f"{type(e).__name__}: {e}"}) + "\n"
        yield json.dumps({"t": "done"}) + "\n"

    return StreamingResponse(gen(), media_type="application/x-ndjson; charset=utf-8")


_config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
_server = uvicorn.Server(_config)
_thread = threading.Thread(target=_server.run, daemon=True)
_thread.start()

_t0 = time.time()
while not _server.started:                        # bounded, and prove the thread is alive
    if not _thread.is_alive():
        raise RuntimeError("uvicorn thread died before binding")
    if time.time() - _t0 > 20:
        raise RuntimeError(f"uvicorn failed to bind port {PORT} within 20s")
    time.sleep(0.1)

BASE_URL = f"http://127.0.0.1:{PORT}"
print(f"serving on {BASE_URL}")
print(f"token     {API_TOKEN}")

import requests
print("health   ", requests.get(f"{BASE_URL}/health", timeout=10).json())


## 11 · Public tunnel (optional)

**cloudflared is the default** because it needs no account. ngrok changed: `ngrok.connect()`
with no authtoken now raises `ERR_NGROK_4018` before a tunnel exists, and the TCP endpoint
that SSH needs requires a card on file even on the free plan.

> ⚠️ A cloudflared quick tunnel is **unauthenticated at the Cloudflare edge** — anyone who
> learns the random hostname reaches this port directly. The bearer token from cell 10 is
> the only thing between the public internet and an A100. It is a fresh
> `secrets.token_urlsafe(24)` per session — but several cells **print** it, and Colab
> autosaves cell outputs into the `.ipynb`. Run `Edit ▸ Clear all outputs` before saving,
> downloading, or committing this notebook. The same goes for the SSH root password.

> ⚠️ Colab's FAQ disallows remote shells on free runtimes with no compute-unit balance and
> may terminate such sessions without warning. An A100 implies paid units, so this is fine
> for the intended user — but do not hand this notebook to someone on the free tier.

The public URL is printed to cloudflared's **log on stderr**, not returned on stdout, so it
has to be launched with output redirected and the hostname polled for.


In [ ]:
import pathlib, re, subprocess, time, urllib.request

def start_cloudflared(port, timeout=60):
    """Quick tunnel -> public https URL. No Cloudflare account needed."""
    if not pathlib.Path("/usr/local/bin/cloudflared").exists():
        subprocess.run(
            "curl -sSL -o /usr/local/bin/cloudflared "
            "https://github.com/cloudflare/cloudflared/releases/latest/download/"
            "cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared",
            shell=True, check=True)

    log = pathlib.Path("/content/cloudflared.log")
    log.write_text("")
    proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--no-autoupdate",
         "--url", f"http://127.0.0.1:{port}"],
        stdout=log.open("ab"), stderr=subprocess.STDOUT)

    url, t0 = None, time.time()
    while time.time() - t0 < timeout:
        hit = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log.read_text())
        if hit:
            url = hit.group(0)
            break
        if proc.poll() is not None:
            raise RuntimeError(f"cloudflared exited:\n{log.read_text()[-2000:]}")
        time.sleep(1)
    if not url:
        raise RuntimeError(f"no tunnel URL in {timeout}s:\n{log.read_text()[-2000:]}")

    # A hostname in the log is not the same as a working edge route. Prove it.
    for _ in range(30):
        try:
            with urllib.request.urlopen(url + "/health", timeout=5) as r:
                if r.status == 200:
                    return url, proc
        except Exception:
            time.sleep(2)
    raise RuntimeError(f"{url} never returned 200 on /health")


PUBLIC_URL, _cf = start_cloudflared(PORT)

print(f"public   {PUBLIC_URL}")
print(f"token    {API_TOKEN}\n")

# A runnable example, not an illustrative one: short slice, and a mutation whose WT letter
# actually matches position 1 of that slice, so copy-pasting it returns a score.
_demo_seq = AV_SEQ[:40]
_demo_mut = f"{_demo_seq[0]}1A"
print("From your laptop:\n")
print(f"  curl -s {PUBLIC_URL}/health\n")
print(f"""  curl -s -X POST {PUBLIC_URL}/score \\
       -H 'Authorization: Bearer {API_TOKEN}' \\
       -H 'Content-Type: application/json' \\
       -d '{{"sequence":"{_demo_seq}","mutations":["{_demo_mut}"]}}'

  # the full 927-residue chain is unwieldy on a command line -- use cli.py for that""")


## 12 · SSH into the VM (optional, needs an ngrok account)

Only worth running if you want a real shell in `/content/conformers` — to run repo scripts,
inspect files, or drive `cli.py` from inside the VM. **The agent and the model are not
reachable from an SSH process**; that shell talks to the notebook over `127.0.0.1:$PORT`
like any other client.

Requires an ngrok authtoken *and* a card on the account (TCP endpoints only). Put the token
in Colab Secrets as `NGROK_AUTH_TOKEN`. Skip this cell entirely if you do not need a shell.

Two things the copy-paste recipe on the web gets wrong:

- `sshd_config` is **first-match-wins**, so appending `PasswordAuthentication yes` with `>>`
  loses to a pre-existing `no` earlier in the file and you get `Permission denied (publickey)`.
  The lines are rewritten with a regex instead.
- Colab has **no systemd**; `systemctl start ssh` fails. Use `service ssh start`, and
  `mkdir -p /var/run/sshd` first or sshd refuses to start with a confusing
  `ssh_exchange_identification` error on the client.


In [ ]:
import os, pathlib, re, secrets, subprocess

ENABLE_SSH = False          # flip to True to open a shell

if not ENABLE_SSH:
    print("SSH disabled. Set ENABLE_SSH = True to enable.")
else:
    try:
        from google.colab import userdata
        ngrok_token = userdata.get("NGROK_AUTH_TOKEN")
    except Exception as e:
        raise RuntimeError(f"Add NGROK_AUTH_TOKEN to Colab Secrets first ({e})")

    subprocess.run("apt-get -qq update && DEBIAN_FRONTEND=noninteractive "
                   "apt-get -qq install -y openssh-server", shell=True, check=True)
    assert pathlib.Path("/usr/sbin/sshd").exists(), "sshd did not install"

    ROOT_PW = secrets.token_urlsafe(18)
    subprocess.run(f"echo 'root:{ROOT_PW}' | chpasswd", shell=True, check=True)

    # first-match-wins: rewrite, never append
    cfg = pathlib.Path("/etc/ssh/sshd_config")
    txt = cfg.read_text()
    for key, val in (("PermitRootLogin", "yes"), ("PasswordAuthentication", "yes")):
        txt = re.sub(rf"^\s*#?\s*{key}\s+.*$", f"{key} {val}", txt, flags=re.M)
        if not re.search(rf"^{key} {val}$", txt, flags=re.M):
            txt += f"\n{key} {val}\n"
    cfg.write_text(txt)

    pathlib.Path("/var/run/sshd").mkdir(parents=True, exist_ok=True)
    subprocess.run("service ssh restart", shell=True, check=True)   # no systemd in Colab

    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(22, "tcp")               # config_version '2' keeps addr/proto
    host, port = tunnel.public_url.replace("tcp://", "").split(":")

    print(f"ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null "
          f"root@{host} -p {port}")
    print(f"password: {ROOT_PW}\n")
    print("Then, inside the VM:")
    print(f"  cd /content/conformers && python cli.py "
          f"--url=http://127.0.0.1:{PORT} --token={API_TOKEN}")


## 13 · The CLI

`cli.py` is a stateless HTTP client — it holds no model, so it starts instantly and can run
anywhere: inside the VM over SSH, or on your laptop against the cloudflared URL.

Colab cells have no tty, so from a notebook only the one-shot `-c` form is usable. The REPL
is for a real terminal.


In [ ]:
import pathlib

CLI_SRC = r"""#!/usr/bin/env python3
# Terminal client for the Colab-hosted ESM-2 3B DeepAgent.
#
#   python cli.py --url URL --token TOK                 # interactive REPL
#   python cli.py --url URL --token TOK -c "question"   # one shot
#   python cli.py --url URL --token TOK --score SEQ K459A E598A
#
# Holds no model: it is a stateless HTTP client, so it starts instantly and runs
# anywhere -- inside the Colab VM over SSH, or on your laptop via the tunnel.
import argparse, json, sys
import requests


def stream_agent(url, token, message, thread):
    r = requests.post(f"{url}/agent", timeout=600, stream=True,
                      headers={"Authorization": f"Bearer {token}"},
                      json={"message": message, "thread_id": thread})
    r.raise_for_status()
    for line in r.iter_lines():          # bytes; json.loads accepts them
        if not line:
            continue
        ev = json.loads(line)
        if ev["t"] == "token":
            sys.stdout.write(ev["v"]); sys.stdout.flush()
        elif ev["t"] == "tool":
            sys.stdout.write(f"\n  [tool {ev['name']}]\n")
        elif ev["t"] == "error":
            sys.stdout.write(f"\n[server error] {ev['v']}\n")
    print()


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--url", required=True)
    p.add_argument("--token", required=True)
    p.add_argument("--thread", default="cli")
    p.add_argument("-c", "--command")
    p.add_argument("--score", nargs="+", metavar=("SEQ", "MUT"))
    a = p.parse_args()
    hdr = {"Authorization": f"Bearer {a.token}"}

    if a.score:
        seq, muts = a.score[0], a.score[1:]
        r = requests.post(f"{a.url}/score", headers=hdr, timeout=600,
                          json={"sequence": seq, "mutations": muts})
        print(json.dumps(r.json(), indent=2))
        return

    if a.command:
        stream_agent(a.url, a.token, a.command, a.thread)
        return

    print(requests.get(f"{a.url}/health", timeout=30).json())
    print("Ctrl-D or 'exit' to quit.\n")
    while True:
        try:
            q = input(">>> ").strip()
        except (EOFError, KeyboardInterrupt):
            print(); return
        if q in ("exit", "quit"):
            return
        if not q:
            continue
        try:
            stream_agent(a.url, a.token, q, a.thread)
        except Exception as e:
            print(f"[client error] {type(e).__name__}: {e}")


if __name__ == "__main__":
    main()
"""

cli_path = REPO / "cli.py"
cli_path.write_text(CLI_SRC)
print(f"wrote {cli_path}")

# One-shot form works from a notebook cell (no tty needed for -c).
# --flag=value, not --flag value: token_urlsafe can start with '-', which argparse would
# otherwise read as an option.
!python {cli_path} --url={BASE_URL} --token={API_TOKEN} --score {AV_SEQ} K459A E598A

_url = globals().get("PUBLIC_URL", BASE_URL)    # cell 11 (tunnel) is optional
print(f"""
From your laptop -- cli.py needs only `requests`, so grab it from GitHub rather than
through the tunnel (the server serves the API, not files):

  curl -sO https://raw.githubusercontent.com/dfu99/conformers/main/cli.py
  python cli.py --url={_url} --token={API_TOKEN}

Until cli.py is committed, copy it out of the Colab file browser instead
(folder icon in the left sidebar -> conformers -> cli.py -> Download).
""")


## 14 · End-to-end validation

Eight checks: GPU allocation, embedding shape, the full 927-residue chain in one pass,
both scoring conventions disagreeing, tool dispatch, an HTTP round-trip, bearer auth
rejecting a bad token, and the agent actually calling an `esm_*` tool.

The last check scores the six genu-lock residues → Ala and lines them up against the MD
lock energies in `results/route_a/lock_energy.json`.

**Read that comparison carefully.** ESM-2 scores are evolutionary — they measure how
surprising a substitution is across the sequence families the model was trained on. The MD
numbers are electrostatic interaction energies in one structure. There is no reason these
must agree, and a low correlation is not a failure of either method. The check is here to
prove the *pipeline* is wired up end to end on real project data; the rank correlation is
reported as an observation, not a validation of the genu-lock hypothesis.


In [ ]:
import json, requests, torch

results, failures = [], []

def check(name, fn):
    try:
        detail = fn()
        results.append((name, "PASS", detail))
    except Exception as e:
        results.append((name, "FAIL", f"{type(e).__name__}: {e}"))
        failures.append(name)

def c_gpu():
    return (f"{torch.cuda.get_device_name(0)}, "
            f"{torch.cuda.memory_allocated()/1e9:.2f} GB model resident")

def c_embed():
    v = extract_embeddings(AV_SEQ[:200])[0]
    assert v.shape[0] == EMBED_DIM
    return f"dim {v.shape[0]}, |v|={float(v.norm()):.1f}"

def c_full_chain():
    v = extract_embeddings(AV_SEQ)[0]
    return f"{len(AV_SEQ)} residues -> dim {v.shape[0]}, one forward pass"

def c_conventions():
    a = score_mutations(AV_SEQ, ["K459A"], "masked")[0]["score"]
    b = score_mutations(AV_SEQ, ["K459A"], "wt")[0]["score"]
    assert abs(a - b) > 1e-3, "conventions gave identical scores -- suspicious"
    return f"masked={a:.3f}  wt={b:.3f}"

def c_tool():
    r = esm_score_mutations.invoke(
        {"sequence": AV_SEQ, "mutations": GENU_MUTATIONS, "method": "masked"})
    assert "error" not in r, r
    return f"{r['n_scored']} scored, most deleterious {r['most_deleterious']}"

def c_http():
    r = requests.post(f"{BASE_URL}/predict", timeout=300,
                      headers={"Authorization": f"Bearer {API_TOKEN}"},
                      json={"sequence": AV_SEQ, "positions": [459, 598], "top_k": 3})
    r.raise_for_status()
    d = r.json()
    return f"{d['n_positions']} positions, top1 at 459 = {d['predictions'][0]['top_k'][0]['aa']}"

def c_auth():
    r = requests.post(f"{BASE_URL}/predict", timeout=60,
                      headers={"Authorization": "Bearer wrong"},
                      json={"sequence": "MKTV", "positions": [2]})
    assert r.status_code == 401, f"bad token got {r.status_code}, not 401"
    return "bad token rejected with 401"

def c_agent():
    ask(f"Using esm_score_mutations with masked-marginals, score "
        f"{', '.join(GENU_MUTATIONS)} on this alphaV sequence and name the two sites the "
        f"model finds most constrained. One short paragraph.\n\n{AV_SEQ}")
    fired = getattr(ask, "last_tools", [])
    assert any(t.startswith("esm_") for t in fired), f"agent called no esm tool: {fired}"
    return f"agent dispatched {fired}"

for _name, _fn in [
    ("gpu allocated", c_gpu),
    ("embedding shape", c_embed),
    ("full alphaV chain in one pass", c_full_chain),
    ("masked vs wt marginals differ", c_conventions),
    ("tool dispatch", c_tool),
    ("http round-trip", c_http),
    ("bearer auth rejects bad token", c_auth),
    ("agent dispatches a tool", c_agent),
]:
    check(_name, _fn)

print(f"{'check':<34} {'status':<6} detail")
print("-" * 96)
for name, status, detail in results:
    print(f"{name:<34} {status:<6} {detail}")

# ---- genu lock: ESM-2 zero-shot vs MD lock energy -------------------------------
scored = score_mutations(AV_SEQ, GENU_MUTATIONS, method="masked")
by_mut = {s["mutation"]: s["score"] for s in scored}

print(f"\n{'mutation':<10} {'ESM-2 score':>12} {'MD kcal removed':>17}  bridges broken")
print("-" * 96)
rows = []
for mut in sorted(by_mut, key=lambda m: by_mut[m]):
    pos = int(mut[1:-1])
    md_kcal = MD_LOCK_KCAL.get(pos)
    bridges = MD_BRIDGES.get(pos, [])
    print(f"{mut:<10} {by_mut[mut]:>12.3f} "
          f"{(f'{md_kcal:.2f}' if md_kcal is not None else '-'):>17}  "
          f"{', '.join(bridges) if bridges else '(not in MD set)'}")
    if md_kcal is not None:
        rows.append((by_mut[mut], md_kcal))

def spearman(pairs):
    """Rank correlation, no scipy. Ties get average ranks."""
    def ranks(xs):
        order = sorted(range(len(xs)), key=lambda i: xs[i])
        r = [0.0] * len(xs)
        i = 0
        while i < len(order):
            j = i
            while j + 1 < len(order) and xs[order[j + 1]] == xs[order[i]]:
                j += 1
            avg = (i + j) / 2 + 1
            for k in range(i, j + 1):
                r[order[k]] = avg
            i = j + 1
        return r
    a, b = ranks([p[0] for p in pairs]), ranks([p[1] for p in pairs])
    n = len(pairs)
    ma, mb = sum(a) / n, sum(b) / n
    num = sum((x - ma) * (y - mb) for x, y in zip(a, b))
    den = (sum((x - ma) ** 2 for x in a) * sum((y - mb) ** 2 for y in b)) ** 0.5
    return num / den if den else float("nan")

if len(rows) >= 3:
    rho = spearman(rows)
    print(f"\nSpearman(ESM-2 score, MD kcal removed) = {rho:+.3f} over n={len(rows)}")
    print("Observation only. ESM-2 measures evolutionary surprise; the MD number is an")
    print("electrostatic interaction energy. Neither validates the other.")

print("\n" + "=" * 96)
if failures:
    raise SystemExit(f"{len(failures)} check(s) failed: {failures}")
print(f"all {len(results)} checks passed")
print(f"server  {PUBLIC_URL if 'PUBLIC_URL' in globals() else BASE_URL}")
print(f"cli     python cli.py --url=<url> --token={API_TOKEN}")
